In [1]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

I0000 00:00:1785483211.594299  697677 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785483211.677247  697677 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785483213.073651  697677 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
train_data = pd.read_csv("../dataset/train_metadata.csv")
val_data = pd.read_csv("../dataset/val_metadata.csv")
test_data = pd.read_csv("../dataset/test_metadata.csv")

In [3]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [4]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [5]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [6]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

E0000 00:00:1785483228.949685  697677 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [7]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.RandomZoom(0.1),

    tf.keras.layers.RandomContrast(0.1)

])

In [8]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [9]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [10]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [11]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

In [12]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

mobilenet_model = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

mobilenet_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [14]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)

In [16]:
history_basic = mobilenet_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 207s 922ms/step - accuracy: 0.3071 - loss: 1.9316 - val_accuracy: 0.4860 - val_loss: 1.5436
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.4100 - loss: 1.6366 - val_accuracy: 0.5140 - val_loss: 1.4464
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 236s 1s/step - accuracy: 0.4506 - loss: 1.5282 - val_accuracy: 0.5772 - val_loss: 1.2977
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.4929 - loss: 1.4491 - val_accuracy: 0.5186 - val_loss: 1.3302
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 232s 1s/step - accuracy: 0.5017 - loss: 1.4070 - val_accuracy: 0.5573 - val_loss: 1.2351
Restoring model weights from the end of the best epoch: 5.


In [17]:
train_loss, train_accuracy = mobilenet_model.evaluate(train_dataset)
val_loss, val_accuracy = mobilenet_model.evaluate(val_dataset)
test_loss, test_accuracy = mobilenet_model.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 186s 821ms/step - accuracy: 0.5719 - loss: 1.2224
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 794ms/step - accuracy: 0.5573 - loss: 1.2351
47/47 ━━━━━━━━━━━━━━━━━━━━ 37s 771ms/step - accuracy: 0.5422 - loss: 1.2909


In [18]:
mobilenet_model.save("../models/mobilenet_model.keras")

In [19]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model_sgd = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model_sgd.trainable = False

mobilenet_model_sgd = models.Sequential([

    base_model_sgd,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

mobilenet_model_sgd.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [20]:
mobilenet_model_sgd.compile(

    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_mobilenet_sgd = mobilenet_model_sgd.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - accuracy: 0.1063 - loss: 2.6370 - val_accuracy: 0.0812 - val_loss: 2.3366
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 222s 986ms/step - accuracy: 0.1350 - loss: 2.4342 - val_accuracy: 0.1025 - val_loss: 2.2036
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 221s 979ms/step - accuracy: 0.1558 - loss: 2.2346 - val_accuracy: 0.1192 - val_loss: 2.1153
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - accuracy: 0.1660 - loss: 2.1247 - val_accuracy: 0.1332 - val_loss: 2.0701
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.1752 - loss: 2.0637 - val_accuracy: 0.1505 - val_loss: 2.0211
Restoring model weights from the end of the best epoch: 5.


In [21]:
train_loss, train_accuracy = mobilenet_model_sgd.evaluate(train_dataset)
val_loss, val_accuracy = mobilenet_model_sgd.evaluate(val_dataset)
test_loss, test_accuracy = mobilenet_model_sgd.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 189s 837ms/step - accuracy: 0.2024 - loss: 1.9201
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 801ms/step - accuracy: 0.1505 - loss: 2.0211
47/47 ━━━━━━━━━━━━━━━━━━━━ 33s 704ms/step - accuracy: 0.1590 - loss: 2.0101


In [22]:
mobilenet_model_sgd.save("../models/mobilenet_model_sgd.keras")

In [23]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model_RMSprop = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model_RMSprop.trainable = False

mobilenet_model_RMSprop= models.Sequential([

    base_model_RMSprop,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

mobilenet_model_RMSprop.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [24]:
mobilenet_model_RMSprop.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_mobilenet_RMSprop = mobilenet_model_RMSprop.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 813ms/step - accuracy: 0.2829 - loss: 1.9842 - val_accuracy: 0.6045 - val_loss: 1.3652
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 183s 807ms/step - accuracy: 0.4248 - loss: 1.6788 - val_accuracy: 0.6332 - val_loss: 1.1823
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 813ms/step - accuracy: 0.4812 - loss: 1.5787 - val_accuracy: 0.6338 - val_loss: 1.1517
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 197s 858ms/step - accuracy: 0.5200 - loss: 1.5056 - val_accuracy: 0.6391 - val_loss: 1.0807
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 823ms/step - accuracy: 0.5198 - loss: 1.4560 - val_accuracy: 0.6751 - val_loss: 1.0059
Restoring model weights from the end of the best epoch: 5.


In [25]:
train_loss, train_accuracy = mobilenet_model_RMSprop.evaluate(train_dataset)
val_loss, val_accuracy = mobilenet_model_RMSprop.evaluate(val_dataset)
test_loss, test_accuracy = mobilenet_model_RMSprop.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 157s 683ms/step - accuracy: 0.6237 - loss: 1.1122
47/47 ━━━━━━━━━━━━━━━━━━━━ 26s 556ms/step - accuracy: 0.6751 - loss: 1.0059
47/47 ━━━━━━━━━━━━━━━━━━━━ 27s 563ms/step - accuracy: 0.6367 - loss: 1.0772


In [26]:
mobilenet_model_RMSprop.save("../models/mobilenet_model_RMSprop.keras")

In [27]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)
 
train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)
 
BATCH_SIZE = 64
 
train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [28]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model_64 = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model_64.trainable = False

mobilenet_64= models.Sequential([

    base_model_sgd,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

mobilenet_64.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [29]:
mobilenet_64.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_mobilenet_64 = mobilenet_64.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.3362 - loss: 1.7574 - val_accuracy: 0.5686 - val_loss: 1.3235
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 187s 2s/step - accuracy: 0.4785 - loss: 1.4937 - val_accuracy: 0.6238 - val_loss: 1.1997
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 190s 2s/step - accuracy: 0.4712 - loss: 1.4106 - val_accuracy: 0.4887 - val_loss: 1.3228
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.4953 - loss: 1.3121 - val_accuracy: 0.6039 - val_loss: 1.1184
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 185s 2s/step - accuracy: 0.5387 - loss: 1.2521 - val_accuracy: 0.5826 - val_loss: 1.1171
Restoring model weights from the end of the best epoch: 5.


In [30]:
train_loss, train_accuracy = mobilenet_64.evaluate(train_dataset)
val_loss, val_accuracy = mobilenet_64.evaluate(val_dataset)
test_loss, test_accuracy = mobilenet_64.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 160s 1s/step - accuracy: 0.5715 - loss: 1.1482
24/24 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.5826 - loss: 1.1171
24/24 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step - accuracy: 0.5449 - loss: 1.1857


In [31]:
mobilenet_64.save("../models/mobilenet_64.keras")

In [32]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load MobileNetV2

base_model = MobileNetV2(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)

# Fine-Tuning

base_model.trainable = True

# Freeze all layers except the last 30

for layer in base_model.layers[:-30]:

    layer.trainable = False

# Build Model

mobilenet_ft = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        7,
        activation="softmax"
    )

])

mobilenet_ft.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 1,691,271 (6.45 MB)

 Non-trainable params: 731,584 (2.79 MB)

In [33]:
mobilenet_ft.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_mobilenet_ft = mobilenet_ft.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 224s 2s/step - accuracy: 0.3355 - loss: 1.7236 - val_accuracy: 0.2597 - val_loss: 1.8410
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 214s 2s/step - accuracy: 0.4979 - loss: 1.2850 - val_accuracy: 0.3589 - val_loss: 1.7191
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 217s 2s/step - accuracy: 0.5636 - loss: 1.1109 - val_accuracy: 0.5213 - val_loss: 1.3208
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 217s 2s/step - accuracy: 0.5917 - loss: 0.9862 - val_accuracy: 0.5320 - val_loss: 1.2935
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 213s 2s/step - accuracy: 0.6138 - loss: 0.8773 - val_accuracy: 0.4201 - val_loss: 1.7118
Restoring model weights from the end of the best epoch: 4.


In [34]:
train_loss, train_accuracy = mobilenet_ft.evaluate(train_dataset)
val_loss, val_accuracy = mobilenet_ft.evaluate(val_dataset)
test_loss, test_accuracy = mobilenet_ft.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 151s 1s/step - accuracy: 0.5136 - loss: 1.3724
24/24 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - accuracy: 0.5320 - loss: 1.2935
24/24 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.5143 - loss: 1.3497


In [35]:
mobilenet_ft.save("../models/mobilenet_ft.keras")

In [12]:
import keras_tuner as kt
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, RMSprop

num_classes = 7

def build_model(hp):

    base_model = MobileNetV2(

        weights="imagenet",

        include_top=False,

        input_shape=(224,224,3)

    )

    base_model.trainable = False

    mobile_model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(

            units=hp.Choice(

                "dense_units",

                [128,256,512]

            ),

            activation="relu"

        ),

        layers.Dropout(

            hp.Choice(

                "dropout",

                [0.3,0.5,0.6]

            )

        ),

        layers.Dense(

            num_classes,

            activation="softmax"

        )

    ])

    learning_rate = hp.Choice(

        "learning_rate",

        [1e-3,1e-4,1e-5]

    )

    optimizer = hp.Choice(

        "optimizer",

        ["adam","rmsprop"]

    )

    if optimizer == "adam":

        opt = Adam(

            learning_rate=learning_rate

        )

    else:

        opt = RMSprop(

            learning_rate=learning_rate

        )

    mobile_model.compile(

        optimizer=opt,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return mobile_model

In [13]:
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=3,

    directory="mobilenet_tuner",

    project_name="mobilenet_hyperparameter"

)

Reloading Tuner from mobilenet_tuner/mobilenet_hyperparameter/tuner0.json


In [14]:
tuner.search(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights

)

Trial 4 Complete [00h 16m 34s]
val_accuracy: 0.6797603368759155

Best val_accuracy So Far: 0.6797603368759155
Total elapsed time: 13h 16m 07s


In [15]:
best_mobilenet = tuner.get_best_models(1)[0]

/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [16]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'dense_units': 128, 'dropout': 0.6, 'learning_rate': 0.001, 'optimizer': 'rmsprop'}


In [17]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

mobilenet_final= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.6),
    layers.Dense(7,activation="softmax")

])

mobilenet_final.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [18]:
mobilenet_final.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_mobilenet_final= mobilenet_final.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 178s 762ms/step - accuracy: 0.3324 - loss: 1.8427 - val_accuracy: 0.5260 - val_loss: 1.3798
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 175s 767ms/step - accuracy: 0.4086 - loss: 1.5908 - val_accuracy: 0.5606 - val_loss: 1.2460
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 169s 738ms/step - accuracy: 0.4415 - loss: 1.4889 - val_accuracy: 0.5293 - val_loss: 1.2526
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 176s 774ms/step - accuracy: 0.4454 - loss: 1.4368 - val_accuracy: 0.6125 - val_loss: 1.1284
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 170s 746ms/step - accuracy: 0.4548 - loss: 1.3895 - val_accuracy: 0.4907 - val_loss: 1.2243
Restoring model weights from the end of the best epoch: 4.


In [20]:
train_loss, train_accuracy = mobilenet_final.evaluate(train_dataset)
val_loss, val_accuracy = mobilenet_final.evaluate(val_dataset)
test_loss, test_accuracy = mobilenet_final.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 148s 647ms/step - accuracy: 0.5749 - loss: 1.2050
47/47 ━━━━━━━━━━━━━━━━━━━━ 28s 579ms/step - accuracy: 0.6125 - loss: 1.1284
47/47 ━━━━━━━━━━━━━━━━━━━━ 26s 551ms/step - accuracy: 0.5749 - loss: 1.1777


In [22]:
mobilenet_final.save("../models/mobilenet_final.keras")